In [11]:
from dataclasses import dataclass
from pydantic.dataclasses import dataclass
from pydantic import ConfigDict
@dataclass(config=ConfigDict(arbitrary_types_allowed=True))
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [12]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
model = ChatOllama(
model="llama3.2:latest", 
)
agent = create_agent(model=model,
context_schema=ColourContext)

In [13]:
from langchain.messages import HumanMessage
response = agent.invoke(
{"messages": [HumanMessage(content="What is my favourite colour?")]},
context=ColourContext()
)
print(response['messages'][-1].content)

I don't have any information about your favourite colour. We just started our conversation, and I haven't asked you any questions yet. Would you like to share your favourite colour with me?


In [14]:
from langchain.tools import tool, ToolRuntime
@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    runtime.context.favourite_colour
@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour
agent = create_agent(
model=model,
tools=[get_favourite_colour, get_least_favourite_colour],
context_schema=ColourContext
)
response = agent.invoke(
{"messages": [HumanMessage(content="What is my favourite colour?")]},
context=ColourContext()

)
print(response['messages'][-1].content)

d:\MASTER\AGENTICAI\AGENTIC-AI-\context-agent\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
d:\MASTER\AGENTICAI\AGENTIC-AI-\context-agent\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


It seems like I wasn't able to retrieve your favourite colour from the tool call response. Unfortunately, this means I don't have enough information to provide an answer to your question. Can you please tell me what your favourite colour is? I'll be happy to chat with you about it!


In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    config={"configurable": {"colour_context": ColourContext(favourite_colour="green", least_favourite_colour="red")}}
)
print(response['messages'][-1].content)


{"name": "get_favourite_colour", "parameters": {"user": {}"}}


In [21]:
from langchain.agents import AgentState
class CustomState(AgentState):
    favourite_colour: str

In [22]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
    "favourite_colour": favourite_colour,
    "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
    )

In [23]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
model=model,

tools=[update_favourite_colour],
checkpointer=InMemorySaver(),
state_schema=CustomState
)

In [24]:
from langchain.messages import HumanMessage
response = agent.invoke(
{ "messages": [HumanMessage(content="My favourite colour is green")]},
{"configurable": {"thread_id": "1"}}
)
print(response['messages'][-1].content)

It seems that I need to provide a favorite color value instead of just saying it's green.

So, my answer is: My favourite colour is indeed green! Do you have a favourite colour too?


In [25]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"
agent = create_agent(
model=model,
tools=[update_favourite_colour, read_favourite_colour],
checkpointer=InMemorySaver(),
state_schema=CustomState
)
response = agent.invoke(
{ "messages": [HumanMessage(content="My favourite colour is green")]},
{"configurable": {"thread_id": "1"}}
)
print(response['messages'][-1].content)

response = agent.invoke(
{ "messages": [HumanMessage(content="What's my favourite colour?")]},
{"configurable": {"thread_id": "1"}}
)
print(response['messages'][-1].content)

So, your favourite colour is indeed green! Would you like to know more about the psychology of different colours or perhaps some fun facts about the colour green?
Your favourite colour is indeed green! Would you like to know more about the psychology of different colours or perhaps some fun facts about the colour green?
